# 02 — Data Cleaning Pipeline
**Objective:** Apply all 20 cleaning rules to the raw CEMS data in a dependency-aware order.  
Every change is logged to `cleaning_log.csv` for a complete audit trail.

We work with the **dev dataset** for fast iteration.

---

## Setup & Load

In [131]:
import pandas as pd
import numpy as np
import re
import os
import hashlib
from datetime import datetime, timedelta

# ── Config ──
DATA_DIR = '../datasets/full'
OUT_DIR  = '../cleaned/full'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load all datasets ──
raw       = pd.read_csv(f'{DATA_DIR}/raw_cems_data.csv')
sensors   = pd.read_csv(f'{DATA_DIR}/sensor_master.csv')
maint     = pd.read_csv(f'{DATA_DIR}/maintenance_logs.csv')
manual    = pd.read_csv(f'{DATA_DIR}/manual_entries.csv')
thresholds = pd.read_csv(f'{DATA_DIR}/regulatory_thresholds.csv')

# work on a copy
df = raw.copy()

# ── Audit log ──
# Every change is recorded: (Record_ID, Column, Old_Value, New_Value, Rule)
audit_log = []

def log_change(record_id, column, old_val, new_val, rule):
    """Append one change to the audit log."""
    audit_log.append({
        'Record_ID': record_id,
        'Column': column,
        'Old': str(old_val),
        'New': str(new_val),
        'Rule': rule
    })

def log_bulk(mask, column, old_series, new_series, rule):
    """Log changes for all rows where mask is True."""
    ids = df.loc[mask, 'Record_ID']
    for idx in ids.index:
        log_change(ids[idx], column, old_series[idx], new_series[idx], rule)

# running summary
rule_summary = []

print(f'Loaded raw_cems_data: {df.shape}')
print(f'Ready to clean.')

Loaded raw_cems_data: (27121, 11)
Ready to clean.


---
## Phase A: Text Normalization
Fix string columns before any parsing.

### Rule 2 — Standardize Unit Strings
Unicode `µg/m³` → plain ASCII `ug/m3`. We fix the string first, unit conversion (mg→ug) comes later in Rule 14.

In [132]:
# ═══ Rule 2: Standardize Unit Strings ═══
# Before
mask_r2 = df['Unit'].isin(['µg/m³'])  # unicode variant
count_r2 = mask_r2.sum()
print(f'Rule 2 — Unicode units found: {count_r2}')

# Fix: normalize to 'ug/m3'
old_vals = df.loc[mask_r2, 'Unit'].copy()
df.loc[mask_r2, 'Unit'] = 'ug/m3'
log_bulk(mask_r2, 'Unit', old_vals, df.loc[mask_r2, 'Unit'], 'R2')

# After
print(f'Fixed: {count_r2} rows → Unit = ug/m3')
print(f'Unit values now: {df["Unit"].value_counts().to_dict()}')
rule_summary.append(('R2', 'Standardize Unit Strings', count_r2))

Rule 2 — Unicode units found: 2623
Fixed: 2623 rows → Unit = ug/m3
Unit values now: {'ug/m3': 24376, 'mg/Nm3': 2745}


### Rule 3 — Trim & Normalize Status Whitespace
Remove leading/trailing whitespace from Status values. Case-normalization to UPPER.

In [133]:
# ═══ Rule 3: Trim & Normalize Status Whitespace ═══
# Before
old_status = df['Status'].copy()
# strip whitespace and uppercase
df['Status'] = df['Status'].astype(str).str.strip().str.upper()

mask_r3 = old_status.astype(str) != df['Status']
count_r3 = mask_r3.sum()
print(f'Rule 3 — Status values trimmed/normalized: {count_r3}')

# Log
log_bulk(mask_r3, 'Status', old_status, df['Status'], 'R3')

# After
print(f'Status values now: {df["Status"].value_counts().to_dict()}')
rule_summary.append(('R3', 'Trim & Normalize Status', count_r3))

Rule 3 — Status values trimmed/normalized: 7742
Status values now: {'OK': 8542, 'MAINT': 7456, 'FAULT': 6903, 'OFFLINE': 1162, 'DOWN': 1147, 'ERROR': 630, 'ERROR 404': 585, 'MAINTENANCE': 574, 'NAN': 122}


---
## Phase B: Datetime Fixes
Fix timestamps before any time-based operations.

### Timestamp Format Standardization
All timestamps must be in `DD-MM-YYYY HH:MM` format. We handle:
- Slash format `DD/MM/YYYY` → `DD-MM-YYYY`
- ISO format `YYYY-MM-DD HH:MM:SS` → `DD-MM-YYYY HH:MM`

In [134]:
# ═══ Timestamp Format Standardization ═══
# Fix slash format: DD/MM/YYYY HH:MM → DD-MM-YYYY HH:MM
mask_slash = df['TS'].str.contains('/', na=False)
count_slash = mask_slash.sum()
old_ts_slash = df.loc[mask_slash, 'TS'].copy()
df.loc[mask_slash, 'TS'] = df.loc[mask_slash, 'TS'].str.replace('/', '-')
log_bulk(mask_slash, 'TS', old_ts_slash, df.loc[mask_slash, 'TS'], 'TS_FORMAT')
print(f'Slash format fixed: {count_slash} rows')

# Fix ISO format: YYYY-MM-DD HH:MM:SS → DD-MM-YYYY HH:MM
mask_iso = df['TS'].str.match(r'^\d{4}-', na=False)
count_iso = mask_iso.sum()
old_ts_iso = df.loc[mask_iso, 'TS'].copy()

def convert_iso_to_standard(ts_str):
    """Convert YYYY-MM-DD HH:MM:SS to DD-MM-YYYY HH:MM"""
    try:
        dt = datetime.strptime(ts_str.strip(), '%Y-%m-%d %H:%M:%S')
        return dt.strftime('%d-%m-%Y %H:%M')
    except:
        return ts_str  # leave as-is if parsing fails

if count_iso > 0:
    df.loc[mask_iso, 'TS'] = df.loc[mask_iso, 'TS'].apply(convert_iso_to_standard)
    log_bulk(mask_iso, 'TS', old_ts_iso, df.loc[mask_iso, 'TS'], 'TS_FORMAT')

print(f'ISO format fixed: {count_iso} rows')
rule_summary.append(('TS_FMT', 'Standardize Timestamp Format', count_slash + count_iso))

Slash format fixed: 1329 rows
ISO format fixed: 545 rows


### Rule 1 — Fix Midnight Rollover (Hour=24)
Timestamps with `24:xx` need to roll over to `00:xx` of the next day.

In [135]:
# ═══ Rule 1: Fix Midnight Rollover ═══
mask_r1 = df['TS'].str.contains(r'\b24:\d{2}', na=False, regex=True)
count_r1 = mask_r1.sum()
print(f'Rule 1 — Midnight rollover timestamps: {count_r1}')

old_ts_r1 = df.loc[mask_r1, 'TS'].copy()

def fix_midnight(ts_str):
    """Fix 24:xx → 00:xx of the next day."""
    # extract date and time parts
    parts = ts_str.strip().split(' ')
    date_str = parts[0]
    time_str = parts[1]  # e.g. '24:15'
    minute = time_str.split(':')[1]
    
    # parse the date and add one day
    dt = datetime.strptime(date_str, '%d-%m-%Y')
    dt = dt + timedelta(days=1)
    
    return dt.strftime('%d-%m-%Y') + f' 00:{minute}'

if count_r1 > 0:
    df.loc[mask_r1, 'TS'] = df.loc[mask_r1, 'TS'].apply(fix_midnight)
    log_bulk(mask_r1, 'TS', old_ts_r1, df.loc[mask_r1, 'TS'], 'R1')

print(f'Fixed: {count_r1} midnight rollovers')
rule_summary.append(('R1', 'Fix Midnight Rollover', count_r1))

Rule 1 — Midnight rollover timestamps: 797
Fixed: 797 midnight rollovers


### Rule 17 — Convert UTC to IST
Timestamps ending with `UTC` need +5:30 to convert to IST.

In [136]:
# ═══ Rule 17: Convert UTC to IST ═══
mask_r17 = df['TS'].str.contains('UTC', na=False)
count_r17 = mask_r17.sum()
print(f'Rule 17 — UTC timestamps: {count_r17}')

old_ts_r17 = df.loc[mask_r17, 'TS'].copy()

def utc_to_ist(ts_str):
    """Convert UTC timestamp to IST (+5:30)."""
    clean = ts_str.replace('UTC', '').strip()
    dt = datetime.strptime(clean, '%d-%m-%Y %H:%M')
    dt = dt + timedelta(hours=5, minutes=30)
    return dt.strftime('%d-%m-%Y %H:%M')

if count_r17 > 0:
    df.loc[mask_r17, 'TS'] = df.loc[mask_r17, 'TS'].apply(utc_to_ist)
    log_bulk(mask_r17, 'TS', old_ts_r17, df.loc[mask_r17, 'TS'], 'R17')

print(f'Fixed: {count_r17} UTC → IST conversions')
rule_summary.append(('R17', 'Convert UTC to IST', count_r17))

Rule 17 — UTC timestamps: 1292
Fixed: 1292 UTC → IST conversions


In [137]:
# Now parse all timestamps into proper datetime
df['TS'] = pd.to_datetime(df['TS'], format='%d-%m-%Y %H:%M')
print(f'All timestamps parsed. dtype: {df["TS"].dtype}')
print(f'Date range: {df["TS"].min()} → {df["TS"].max()}')

All timestamps parsed. dtype: datetime64[ns]
Date range: 2025-01-01 00:00:00 → 2025-01-30 23:45:00


---
## Phase C: Deduplication & Gap Filling

### Rule 6 — Remove Duplicates
Drop exact duplicate rows (same Plant_ID + Stack_ID + TS).

In [138]:
# ═══ Rule 6: Remove Duplicates ═══
before_count = len(df)
df = df.drop_duplicates(subset=['Plant_ID', 'Stack_ID', 'TS'], keep='first')
count_r6 = before_count - len(df)
print(f'Rule 6 — Duplicates removed: {count_r6}')
rule_summary.append(('R6', 'Remove Duplicates', count_r6))

Rule 6 — Duplicates removed: 736


### Rule 9 — Fill Gaps with Placeholder Rows
Check for missing Record_IDs and insert sentinel rows with NaN values.

In [139]:
# ═══ Rule 9: Fill Record_ID Gaps ═══
record_nums = df['Record_ID'].str.replace('E', '').astype(int)
expected = set(range(record_nums.min(), record_nums.max() + 1))
actual = set(record_nums)
missing_ids = sorted(expected - actual)
count_r9 = len(missing_ids)
print(f'Rule 9 — Missing Record_IDs: {count_r9}')

# Create placeholder rows for each missing ID
if count_r9 > 0:
    gap_rows = []
    for mid in missing_ids:
        gap_rows.append({
            'Record_ID': f'E{mid:05d}',
            'Plant_ID': np.nan,
            'Stack_ID': np.nan,
            'Flow_Rate_m3_hr': np.nan,
            'TS': pd.NaT,
            'PM2.5': np.nan,
            'SO2': np.nan,
            'NOx': np.nan,
            'Unit': 'ug/m3',
            'Status': 'GAP_FILLED',
            'Lat_Lon': np.nan
        })
        log_change(f'E{mid:05d}', 'ALL', 'MISSING', 'GAP_FILLED_ROW', 'R9')
    
    gap_df = pd.DataFrame(gap_rows)
    df = pd.concat([df, gap_df], ignore_index=True)
    df = df.sort_values(by='Record_ID').reset_index(drop=True)

print(f'Inserted: {count_r9} placeholder rows. Total rows now: {len(df)}')
rule_summary.append(('R9', 'Fill Record_ID Gaps', count_r9))

Rule 9 — Missing Record_IDs: 2415
Inserted: 2415 placeholder rows. Total rows now: 28800


---
## Phase D: Value Cleaning
Fix pollutant values — remove negatives, handle BDL, convert units.

### Rule 5 — Discard Negative Pollutant Values
Negative concentrations are physically impossible. Replace with NaN.

In [140]:
# ═══ Rule 5: Discard Negative Pollutants ═══
total_r5 = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    numeric = pd.to_numeric(df[col], errors='coerce')
    mask = numeric < 0
    count = mask.sum()
    if count > 0:
        old_vals = df.loc[mask, col].copy()
        df.loc[mask, col] = np.nan
        log_bulk(mask, col, old_vals, df.loc[mask, col], 'R5')
    total_r5 += count
    print(f'  {col}: {count} negatives → NaN')

print(f'Rule 5 — Total negatives removed: {total_r5}')
rule_summary.append(('R5', 'Discard Negative Pollutants', total_r5))

  PM2.5: 1354 negatives → NaN
  SO2: 1351 negatives → NaN
  NOx: 1356 negatives → NaN
Rule 5 — Total negatives removed: 4061


### Rule 11 — Handle Below Detection Limit (BDL)
Replace BDL strings (`<2.0`, `BDL`, `< 5.0`) with `LOD / 2` using sensor_master.

In [141]:
# ═══ Rule 11: Handle BDL Values ═══
# Build a LOD lookup from sensor_master: (Plant_ID, Stack_ID) → {PM2.5: lod, SO2: lod, NOx: lod}
lod_lookup = {}
for _, row in sensors.iterrows():
    key = (row['Plant_ID'], row['Stack_ID'])
    lod_lookup[key] = {
        'PM2.5': row['LOD_PM25'],
        'SO2': row['LOD_SO2'],
        'NOx': row['LOD_NOx']
    }

total_r11 = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    # find BDL entries: starts with '<' or equals 'BDL'
    is_bdl = df[col].astype(str).str.strip().str.upper().isin(['BDL']) | \
             df[col].astype(str).str.strip().str.startswith('<')
    count = is_bdl.sum()
    
    if count > 0:
        old_vals = df.loc[is_bdl, col].copy()
        # replace each BDL with LOD/2 from its sensor
        for idx in df.loc[is_bdl].index:
            key = (df.at[idx, 'Plant_ID'], df.at[idx, 'Stack_ID'])
            lod_val = lod_lookup.get(key, {}).get(col, 2.0)
            replacement = round(lod_val / 2, 2)
            df.at[idx, col] = str(replacement)
            log_change(df.at[idx, 'Record_ID'], col, old_vals[idx], str(replacement), 'R11')
    
    total_r11 += count
    print(f'  {col}: {count} BDL → LOD/2')

print(f'Rule 11 — Total BDL replaced: {total_r11}')
rule_summary.append(('R11', 'Handle BDL Values', total_r11))

  PM2.5: 1352 BDL → LOD/2
  SO2: 1350 BDL → LOD/2
  NOx: 1359 BDL → LOD/2
Rule 11 — Total BDL replaced: 4061


### Rule 14 — Convert mg/Nm3 to ug/m3
Rows with `Unit = mg/Nm3` need pollutant values multiplied by 1000, then unit changed to `ug/m3`.

In [142]:
# ═══ Rule 14: Convert mg/Nm3 → ug/m3 ═══
mask_r14 = df['Unit'] == 'mg/Nm3'
count_r14 = mask_r14.sum()
print(f'Rule 14 — Rows with mg/Nm3: {count_r14}')

if count_r14 > 0:
    for col in ['PM2.5', 'SO2', 'NOx']:
        old_vals = df.loc[mask_r14, col].copy()
        # convert to numeric, multiply by 1000
        numeric = pd.to_numeric(df.loc[mask_r14, col], errors='coerce')
        df.loc[mask_r14, col] = (numeric * 1000).round(1).astype(str)
        log_bulk(mask_r14, col, old_vals, df.loc[mask_r14, col], 'R14')
    
    # fix the unit
    old_units = df.loc[mask_r14, 'Unit'].copy()
    df.loc[mask_r14, 'Unit'] = 'ug/m3'
    log_bulk(mask_r14, 'Unit', old_units, df.loc[mask_r14, 'Unit'], 'R14')

print(f'Converted: {count_r14} rows from mg/Nm3 → ug/m3')
rule_summary.append(('R14', 'Convert mg/Nm3 to ug/m3', count_r14))

Rule 14 — Rows with mg/Nm3: 2664
Converted: 2664 rows from mg/Nm3 → ug/m3


In [143]:
# Now convert all pollutant columns to numeric (float)
for col in ['PM2.5', 'SO2', 'NOx']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Pollutant columns converted to float64.')
print(df[['PM2.5', 'SO2', 'NOx']].describe())

Pollutant columns converted to float64.
              PM2.5           SO2           NOx
count  2.464700e+04  2.463300e+04  2.462000e+04
mean   4.092993e+04  4.203104e+04  4.011559e+04
std    4.917760e+05  4.986076e+05  4.957100e+05
min    5.000000e-01  1.000000e+00  1.000000e+00
25%    8.430000e+01  8.520000e+01  8.460000e+01
50%    1.322000e+02  1.323000e+02  1.319000e+02
75%    1.865000e+02  1.846000e+02  1.846000e+02
max    9.999000e+06  9.999000e+06  9.999000e+06


---
## Phase E: Calibration

### Rule 7 — Apply Zero Drift & Span Correction
Using `sensor_master`, adjust each sensor's readings:  
`corrected = (raw - Zero_Drift) / Span_Mult`

In [144]:
# ═══ Rule 7: Calibration (Drift & Span) ═══
# Build calibration lookup
cal_lookup = {}
for _, row in sensors.iterrows():
    key = (row['Plant_ID'], row['Stack_ID'])
    cal_lookup[key] = {
        'zero_drift': row['Zero_Drift'],
        'span_mult': row['Span_Mult']
    }

total_r7 = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    for idx in df.index:
        key = (df.at[idx, 'Plant_ID'], df.at[idx, 'Stack_ID'])
        cal = cal_lookup.get(key)
        if cal is None or pd.isna(df.at[idx, col]):
            continue
        
        old_val = df.at[idx, col]
        drift = cal['zero_drift']
        span = cal['span_mult']
        
        # only calibrate if drift or span are non-trivial
        if drift != 0 or span != 1.0:
            new_val = round((old_val - drift) / span, 2)
            df.at[idx, col] = new_val
            log_change(df.at[idx, 'Record_ID'], col, old_val, new_val, 'R7')
            total_r7 += 1

print(f'Rule 7 — Calibration adjustments: {total_r7}')
rule_summary.append(('R7', 'Calibration (Drift & Span)', total_r7))

Rule 7 — Calibration adjustments: 73900


---
## Phase F: Outlier Detection

### Rule 8 — Cap/Flag Physical Spikes (Hampel)
Values > 5000 are physical impossibilities. Cap at the 99th percentile.

In [145]:
# ═══ Rule 8: Cap Physical Spikes ═══
total_r8 = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    # find spikes > 5000
    mask = df[col] > 5000
    count = mask.sum()
    
    if count > 0:
        # cap at 99th percentile of valid data
        valid = df.loc[(df[col].notna()) & (df[col] <= 5000), col]
        cap_val = round(valid.quantile(0.99), 1)
        
        old_vals = df.loc[mask, col].copy()
        df.loc[mask, col] = cap_val
        log_bulk(mask, col, old_vals, df.loc[mask, col], 'R8')
        print(f'  {col}: {count} spikes capped at {cap_val}')
    
    total_r8 += count

print(f'Rule 8 — Total spikes capped: {total_r8}')
rule_summary.append(('R8', 'Cap Physical Spikes', total_r8))

  PM2.5: 3069 spikes capped at 273.8
  SO2: 3124 spikes capped at 275.7
  NOx: 3107 spikes capped at 267.0
Rule 8 — Total spikes capped: 9300


---
## Phase G: Geo Validation

### Rule 4 — Fix Invalid Coordinates
Fix semicolon separators, impossible values, swapped lat/lon.

In [146]:
# ═══ Rule 4: Fix Invalid Coordinates ═══
total_r4 = 0

# India bounds (approximate)
LAT_MIN, LAT_MAX = 6.0, 37.0
LON_MIN, LON_MAX = 68.0, 97.5

# Build expected coords from sensor_master
expected_coords = {}
for _, row in sensors.iterrows():
    # sensor_master doesn't store lat/lon, but the generator used known values
    # we'll use the mode of valid coordinates per plant/stack as the reference
    pass

for idx in df.index:
    ll = df.at[idx, 'Lat_Lon']
    if pd.isna(ll):
        continue
    
    ll_str = str(ll).strip()
    old_val = ll_str
    fixed = False
    
    # fix semicolon separator
    if ';' in ll_str:
        ll_str = ll_str.replace(';', ',')
        fixed = True
    
    # parse
    try:
        parts = ll_str.split(',')
        lat, lon = float(parts[0]), float(parts[1])
        
        # check impossible values
        if lat < LAT_MIN or lat > LAT_MAX or lon < LON_MIN or lon > LON_MAX:
            # try swapping
            if LON_MIN <= lat <= LON_MAX and LAT_MIN <= lon <= LAT_MAX:
                lat, lon = lon, lat
                ll_str = f'{lat},{lon}'
                fixed = True
            else:
                # can't fix — mark as NaN
                df.at[idx, 'Lat_Lon'] = np.nan
                log_change(df.at[idx, 'Record_ID'], 'Lat_Lon', old_val, 'NaN', 'R4')
                total_r4 += 1
                continue
        
        if fixed:
            df.at[idx, 'Lat_Lon'] = f'{lat},{lon}'
            log_change(df.at[idx, 'Record_ID'], 'Lat_Lon', old_val, df.at[idx, 'Lat_Lon'], 'R4')
            total_r4 += 1
    except:
        df.at[idx, 'Lat_Lon'] = np.nan
        log_change(df.at[idx, 'Record_ID'], 'Lat_Lon', old_val, 'NaN', 'R4')
        total_r4 += 1

print(f'Rule 4 — Coordinates fixed/nulled: {total_r4}')
rule_summary.append(('R4', 'Fix Invalid Coordinates', total_r4))

Rule 4 — Coordinates fixed/nulled: 514


---
## Phase H: Status Mapping

### Rule 12 — Map Non-Standard Status to Canonical
Map values like `OFFLINE`, `DOWN`, `ERROR 404`, `MAINTENANCE` to `OK`, `FAULT`, or `MAINT`.

In [147]:
# ═══ Rule 12: Map Non-Standard Status ═══
STATUS_MAP = {
    'OFFLINE': 'FAULT',
    'DOWN': 'FAULT',
    'ERROR 404': 'FAULT',
    'ERROR': 'FAULT',
    'MAINTENANCE': 'MAINT',
    'NAN': 'UNKNOWN',
}

canonical = {'OK', 'FAULT', 'MAINT', 'GAP_FILLED', 'UNKNOWN'}
mask_r12 = ~df['Status'].isin(canonical)
count_r12 = mask_r12.sum()
print(f'Rule 12 — Non-standard statuses: {count_r12}')

if count_r12 > 0:
    old_status = df.loc[mask_r12, 'Status'].copy()
    df.loc[mask_r12, 'Status'] = df.loc[mask_r12, 'Status'].map(
        lambda s: STATUS_MAP.get(s, 'FAULT')  # default unmapped to FAULT
    )
    log_bulk(mask_r12, 'Status', old_status, df.loc[mask_r12, 'Status'], 'R12')

print(f'Mapped: {count_r12} non-standard statuses')
print(f'Status values now: {df["Status"].value_counts().to_dict()}')
rule_summary.append(('R12', 'Map Non-Standard Status', count_r12))

Rule 12 — Non-standard statuses: 4114
Mapped: 4114 non-standard statuses
Status values now: {'FAULT': 10157, 'OK': 8313, 'MAINT': 7795, 'GAP_FILLED': 2415, 'UNKNOWN': 120}


---
## Phase I: Cross-Table Joins

### Rule 13 — Validate Plant+Stack Combos
Every (Plant_ID, Stack_ID) in raw data must exist in sensor_master.

In [148]:
# ═══ Rule 13: Validate Plant+Stack Combos ═══
valid_combos = set(zip(sensors['Plant_ID'], sensors['Stack_ID']))
df_combos = set(zip(df['Plant_ID'].dropna(), df['Stack_ID'].dropna()))
invalid = df_combos - valid_combos

print(f'Rule 13 — Invalid Plant+Stack combos: {len(invalid)}')
if invalid:
    print(f'  Invalid combos: {invalid}')
rule_summary.append(('R13', 'Validate Plant+Stack Combos', len(invalid)))

Rule 13 — Invalid Plant+Stack combos: 0


### Rule 15 — Tag Source Type
Add a `Source_Type` column from sensor_master (Stack vs Ambient).

In [149]:
# ═══ Rule 15: Tag Source Type ═══
source_lookup = dict(zip(
    zip(sensors['Plant_ID'], sensors['Stack_ID']),
    sensors['Source_Type']
))

df['Source_Type'] = df.apply(
    lambda row: source_lookup.get((row['Plant_ID'], row['Stack_ID']), 'Unknown'),
    axis=1
)

print(f'Rule 15 — Source_Type column added')
print(f'Source types: {df["Source_Type"].value_counts().to_dict()}')
rule_summary.append(('R15', 'Tag Source Type', len(df)))

Rule 15 — Source_Type column added
Source types: {'Stack': 15801, 'Ambient': 10584, 'Unknown': 2415}


### Rule 10 — Apply Maintenance Windows
If sensor was under maintenance (from maintenance_logs), null pollutant data and set Status=MAINT.

In [150]:
# ═══ Rule 10: Apply Maintenance Windows ═══
total_r10 = 0

# parse maintenance timestamps
maint['Maint_Start'] = pd.to_datetime(maint['Maint_Start'], format='%d-%m-%Y %H:%M')
maint['Maint_End'] = pd.to_datetime(maint['Maint_End'], format='%d-%m-%Y %H:%M')

for _, m_row in maint.iterrows():
    # find rows for this sensor during maintenance window
    mask = (
        (df['Plant_ID'] == m_row['Plant_ID']) &
        (df['Stack_ID'] == m_row['Stack_ID']) &
        (df['TS'] >= m_row['Maint_Start']) &
        (df['TS'] <= m_row['Maint_End']) &
        (df['Status'] != 'MAINT')
    )
    
    count = mask.sum()
    if count > 0:
        # null pollutant data and set status
        for col in ['PM2.5', 'SO2', 'NOx']:
            old_vals = df.loc[mask, col].copy()
            df.loc[mask, col] = np.nan
            log_bulk(mask, col, old_vals, df.loc[mask, col], 'R10')
        
        old_status = df.loc[mask, 'Status'].copy()
        df.loc[mask, 'Status'] = 'MAINT'
        log_bulk(mask, 'Status', old_status, df.loc[mask, 'Status'], 'R10')
        
        total_r10 += count

print(f'Rule 10 — Rows affected by maintenance windows: {total_r10}')
rule_summary.append(('R10', 'Apply Maintenance Windows', total_r10))

Rule 10 — Rows affected by maintenance windows: 199


---
## Phase J: QC & Compliance

### Rule 16 — Double-Entry QC Check (Manual Entries)
If Entry1 and Entry2 differ by >1%, flag as QC_FAIL.

In [151]:
# ═══ Rule 16: Double-Entry QC Check ═══
manual['Diff_Pct'] = abs(manual['Lab_PM25_Entry1'] - manual['Lab_PM25_Entry2']) / manual['Lab_PM25_Entry1'] * 100
manual['QC_Status'] = manual['Diff_Pct'].apply(lambda d: 'QC_FAIL' if d > 1.0 else 'QC_PASS')

qc_fails = (manual['QC_Status'] == 'QC_FAIL').sum()
print(f'Rule 16 — QC failures (>1% difference): {qc_fails} out of {len(manual)}')
print(manual[['Log_ID', 'Lab_PM25_Entry1', 'Lab_PM25_Entry2', 'Diff_Pct', 'QC_Status']])
rule_summary.append(('R16', 'Double-Entry QC Check', qc_fails))

Rule 16 — QC failures (>1% difference): 10 out of 50
   Log_ID  Lab_PM25_Entry1  Lab_PM25_Entry2    Diff_Pct QC_Status
0   L0001             98.7            987.0  900.000000   QC_FAIL
1   L0002            156.4            156.7    0.191816   QC_PASS
2   L0003             49.6             49.2    0.806452   QC_PASS
3   L0004            123.7            124.2    0.404204   QC_PASS
4   L0005            132.9            132.8    0.075245   QC_PASS
5   L0006             69.9            699.0  900.000000   QC_FAIL
6   L0007             86.8             87.1    0.345622   QC_PASS
7   L0008            149.0            149.3    0.201342   QC_PASS
8   L0009            122.2            122.0    0.163666   QC_PASS
9   L0010             44.4             44.5    0.225225   QC_PASS
10  L0011             17.4            174.0  900.000000   QC_FAIL
11  L0012            149.1            148.9    0.134138   QC_PASS
12  L0013             90.2             90.1    0.110865   QC_PASS
13  L0014             4

### Rule 19 — Flag Threshold Exceedances
Compare pollutant values against regulatory_thresholds. Tag exceedances.

In [152]:
# ═══ Rule 19: Flag Threshold Exceedances ═══
# Build threshold lookup: (pollutant, source_type) → limit
limit_lookup = {}
for _, t_row in thresholds.iterrows():
    limit_lookup[(t_row['Pollutant'], t_row['Source_Type'])] = t_row['Legal_Limit_ugm3']

# Add Exceedance_Flag column
df['Exceedance_Flag'] = 'OK'

total_r19 = 0
for col in ['PM2.5', 'SO2', 'NOx']:
    for idx in df.index:
        val = df.at[idx, col]
        src = df.at[idx, 'Source_Type']
        if pd.isna(val) or src == 'Unknown':
            continue
        
        limit = limit_lookup.get((col, src))
        if limit and val > limit:
            df.at[idx, 'Exceedance_Flag'] = 'EXCEEDANCE'
            total_r19 += 1

print(f'Rule 19 — Threshold exceedances flagged: {total_r19}')
print(f'Exceedance flags: {df["Exceedance_Flag"].value_counts().to_dict()}')
rule_summary.append(('R19', 'Flag Threshold Exceedances', total_r19))

Rule 19 — Threshold exceedances flagged: 31872
Exceedance flags: {'EXCEEDANCE': 16983, 'OK': 11817}


---
## Phase K: Privacy

### Rule 18 — Redact PII from Inspection Notes
Replace phone numbers and email addresses with `[REDACTED]`.

In [153]:
# ═══ Rule 18: PII Redaction in Manual Entries ═══
phone_pattern = r'\b\d{10}\b'
email_pattern = r'\S+@\S+\.\S+'

total_r18 = 0
for idx in manual.index:
    note = str(manual.at[idx, 'Inspection_Notes'])
    old_note = note
    
    # redact phone numbers
    note = re.sub(phone_pattern, '[REDACTED_PHONE]', note)
    # redact emails
    note = re.sub(email_pattern, '[REDACTED_EMAIL]', note)
    
    if note != old_note:
        manual.at[idx, 'Inspection_Notes'] = note
        total_r18 += 1

print(f'Rule 18 — Notes with PII redacted: {total_r18}')
rule_summary.append(('R18', 'PII Redaction', total_r18))

Rule 18 — Notes with PII redacted: 16


---
## Phase L: Audit Trail

### Rule 20 — Generate SHA-256 Integrity Hash
Each row gets a hash of its content as a tamper-proof seal.

In [154]:
# ═══ Rule 20: SHA-256 Audit Hash ═══
def row_hash(row):
    """Create a SHA-256 hash of the row's content."""
    content = '|'.join(str(v) for v in row.values)
    return hashlib.sha256(content.encode('utf-8')).hexdigest()

df['Audit_Hash'] = df.apply(row_hash, axis=1)

print(f'Rule 20 — Audit hashes generated for {len(df)} rows')
print(f'Sample hash: {df["Audit_Hash"].iloc[0][:32]}...')
rule_summary.append(('R20', 'SHA-256 Audit Hash', len(df)))

Rule 20 — Audit hashes generated for 28800 rows
Sample hash: d71814caf8869de73c7cb0f4961d17dc...


---
## Save Results

In [155]:
# ═══ Save cleaned data ═══
df.to_csv(f'{OUT_DIR}/raw_cems_data_cleaned.csv', index=False)
manual.to_csv(f'{OUT_DIR}/manual_entries_cleaned.csv', index=False)
sensors.to_csv(f'{OUT_DIR}/sensor_master.csv', index=False)
maint.to_csv(f'{OUT_DIR}/maintenance_logs.csv', index=False)
thresholds.to_csv(f'{OUT_DIR}/regulatory_thresholds.csv', index=False)

# ═══ Save audit log ═══
audit_df = pd.DataFrame(audit_log)
audit_df.to_csv(f'{OUT_DIR}/cleaning_log.csv', index=False)

print(f'\nCleaned data saved to: {OUT_DIR}/')
print(f'Cleaning log entries: {len(audit_df)}')


Cleaned data saved to: ../cleaned/full/
Cleaning log entries: 124145


---
## Cleaning Summary

In [156]:
# ═══ Print Summary ═══
print('=' * 60)
print('  CLEANING PIPELINE — SUMMARY')
print('=' * 60)
print(f'  Original rows:  {len(raw):,}')
print(f'  Cleaned rows:   {len(df):,}')
print(f'  Audit log entries: {len(audit_log):,}')
print('-' * 60)
print(f'{"Rule":<10} {"Description":<35} {"Affected":>10}')
print('-' * 60)
for rule, desc, count in rule_summary:
    print(f'{rule:<10} {desc:<35} {count:>10,}')
print('=' * 60)

  CLEANING PIPELINE — SUMMARY
  Original rows:  27,121
  Cleaned rows:   28,800
  Audit log entries: 124,145
------------------------------------------------------------
Rule       Description                           Affected
------------------------------------------------------------
R2         Standardize Unit Strings                 2,623
R3         Trim & Normalize Status                  7,742
TS_FMT     Standardize Timestamp Format             1,874
R1         Fix Midnight Rollover                      797
R17        Convert UTC to IST                       1,292
R6         Remove Duplicates                          736
R9         Fill Record_ID Gaps                      2,415
R5         Discard Negative Pollutants              4,061
R11        Handle BDL Values                        4,061
R14        Convert mg/Nm3 to ug/m3                  2,664
R7         Calibration (Drift & Span)              73,900
R8         Cap Physical Spikes                      9,300
R4         Fix 